# Final Evaluation (Validation + Test)

This notebook compares the focal model in two modes:
- Classifier head (standard softmax)
- Embedding k-NN (head removed or ignored)

ArcFace and Triplet runs are evaluated only with k-NN.


## Setup and Imports

Load evaluation utilities and register the experiment checkpoints to run.


In [ ]:
import numpy as np
import torch
from pathlib import Path

from config import ExperimentConfig
from dataset import (
    build_dataloaders,
    build_retrieval_eval_loaders,
    build_test_loader,
    prepare_data,
)
from final_evaluation import (
    calculate_map5,
    evaluate_base_metrics,
    get_classifier_predictions,
    get_retrieval_predictions,
    insert_new_whale,
    sweep_thresholds,
)
from models import build_model
from visualization import plot_evaluation, plot_evaluation_pair

# Experiment artifacts to evaluate.
experiments = {
    "focal": {
        "label": "Focal",
        "config": "experiments/20260402_175215_baseline_effnetb5_focal_bd5dccca/config.json",
        "checkpoint": "experiments/20260402_175215_baseline_effnetb5_focal_bd5dccca/checkpoints/best.pth",
    },
    "arcface": {
        "label": "ArcFace",
        "config": "experiments/20260403_163615_arcface_effnetb5_6c8d152f/config.json",
        "checkpoint": "experiments/20260403_163615_arcface_effnetb5_6c8d152f/checkpoints/best.pth",
    },
    "triplet": {
        "label": "Triplet",
        "config": "experiments/20260404_143337_triplet_effnetb5_3ae3db53/config.json",
        "checkpoint": "experiments/20260404_143337_triplet_effnetb5_3ae3db53/checkpoints/best.pth",
    },
}

def is_experiment_available(info: dict) -> bool:
    return Path(info["config"]).exists() and Path(info["checkpoint"]).exists()


## Loaders and Checkpoints

Helper to load config, data splits, loaders, and model weights per experiment.
Supports optional head override for k-NN-only evaluation.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_experiment(info, head_override: str | None = None):
    # Load config and prepare eval loaders.
    config = ExperimentConfig.load(info["config"])
    config.include_new_whale_in_val = True

    if head_override is not None:
        config.head_type = head_override

    data = prepare_data(config)
    _, val_loader = build_dataloaders(config, data)
    gallery_loader, _ = build_retrieval_eval_loaders(config, data)
    test_loader = build_test_loader(config, data)

    # Load model weights.
    checkpoint = torch.load(info["checkpoint"], map_location=device, weights_only=False)
    extra = checkpoint.get("extra", {})
    num_classes = extra.get("num_classes", data["num_classes"])

    model = build_model(config, num_classes=num_classes, device=device)
    state_dict = checkpoint["model_state_dict"]
    if head_override is None:
        model.load_state_dict(state_dict)
    else:
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        if missing or unexpected:
            print(f"  Load info: missing={len(missing)}, unexpected={len(unexpected)}")
    model.eval()

    return config, data, model, gallery_loader, val_loader, test_loader


## Validation Sweep

Find the best open-set threshold on the validation split for each run.


In [ ]:
runs = [
    {
        "run_id": "focal_cls",
        "exp_key": "focal",
        "mode": "classifier",
        "head_override": "linear",
        "label": "Focal - Classifier",
    },
    {
        "run_id": "focal_knn",
        "exp_key": "focal",
        "mode": "knn",
        "head_override": None,
        "label": "Focal - kNN",
    },
    {
        "run_id": "arcface_knn",
        "exp_key": "arcface",
        "mode": "knn",
        "head_override": None,
        "label": "ArcFace - kNN",
    },
    {
        "run_id": "triplet_knn",
        "exp_key": "triplet",
        "mode": "knn",
        "head_override": None,
        "label": "Triplet - kNN",
    },
]

results = {}


In [ ]:
for run in runs:
    info = experiments.get(run["exp_key"])
    if info is None or not is_experiment_available(info):
        print(f"Skipping {run['run_id']}: missing config or checkpoint")
        continue

    print(f"\n=== {run['label']} ===")
    config, data, model, gallery_loader, val_loader, _ = load_experiment(
        info, head_override=run["head_override"]
    )

    # Validation predictions for threshold sweep.
    if run["mode"] == "classifier":
        if config.head_type == "none":
            print("  Skipping classifier mode because head_type is 'none'")
            continue
        scores, preds, labels = get_classifier_predictions(model, val_loader, device)
    else:
        scores, preds, labels = get_retrieval_predictions(
            model, gallery_loader, val_loader, device
        )

    base_metrics = evaluate_base_metrics(scores, preds, labels)
    thresholds, map_scores, best_t, best_map = sweep_thresholds(scores, preds, labels)

    results[run["run_id"]] = {
        "label": run["label"],
        "mode": run["mode"],
        "exp_key": run["exp_key"],
        "base_metrics": base_metrics,
        "best_threshold": best_t,
        "best_map5": best_map,
        "thresholds": thresholds,
        "map_scores": map_scores,
    }

    print("Base Validation Metrics:", base_metrics)
    print(f"Optimal Validation Threshold: {best_t:.3f} -> MAP@5: {best_map:.4f}")


## Test Evaluation

Apply the validation threshold to test and report the final MAP@5.
Classifier vs k-NN plots are stacked for the focal run.


In [ ]:
for run in runs:
    if run["run_id"] not in results:
        continue
    info = experiments[run["exp_key"]]
    print(f"\n=== {run['label']} (test) ===")
    config, data, model, gallery_loader, _, test_loader = load_experiment(
        info, head_override=run["head_override"]
    )

    if run["mode"] == "classifier":
        if config.head_type == "none":
            print("  Skipping classifier mode because head_type is 'none'")
            continue
        test_scores, test_preds, test_labels = get_classifier_predictions(
            model, test_loader, device
        )
    else:
        test_scores, test_preds, test_labels = get_retrieval_predictions(
            model, gallery_loader, test_loader, device
        )

    base_test_metrics = evaluate_base_metrics(test_scores, test_preds, test_labels)

    test_preds_np = test_preds.cpu().numpy()
    test_scores_np = test_scores.cpu().numpy()
    test_labels_np = test_labels.cpu().numpy()
    best_threshold = results[run["run_id"]]["best_threshold"]

    # Apply validation threshold to test predictions.
    final_test_preds = np.array([
        insert_new_whale(p, s, best_threshold)
        for p, s in zip(test_preds_np, test_scores_np)
    ])

    final_test_map5 = calculate_map5(final_test_preds, test_labels_np)

    results[run["run_id"]]["test_base_metrics"] = base_test_metrics
    results[run["run_id"]]["final_test_map5"] = final_test_map5
    results[run["run_id"]]["test_scores"] = test_scores
    results[run["run_id"]]["test_labels"] = test_labels

    print("Test Base Metrics:", base_test_metrics)
    print(f"Final Test MAP@5: {final_test_map5:.4f}")

    plot_evaluation(
        scores=test_scores,
        labels=test_labels,
        thresholds=results[run["run_id"]]["thresholds"],
        map5_scores=results[run["run_id"]]["map_scores"],
        best_threshold=best_threshold,
        best_map5=final_test_map5,
        model_name=run["label"],
    )

if "focal_cls" in results and "focal_knn" in results:
    plot_evaluation_pair(
        results["focal_cls"],
        results["focal_knn"],
        title="Focal: Classifier vs kNN",
    )
